# COVID-19 Global Data Analysis
### Methodology: Data Collection → Preprocessing → EDA → SIR Model → Evaluation → Visualization

## 1. Imports & Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.integrate import odeint
from scipy.optimize import minimize
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('tab10')
print('All libraries loaded successfully.')

All libraries loaded successfully.


## 2. Data Collection

In [ ]:
# The CONVENIENT format stores daily new cases/deaths.
# Each column is a country/region; each row (after the header rows) is a date.
cases_raw  = pd.read_csv('CONVENIENT_global_confirmed_cases.csv')
deaths_raw = pd.read_csv('CONVENIENT_global_deaths.csv')

print(f'Cases  raw shape : {cases_raw.shape}')
print(f'Deaths raw shape : {deaths_raw.shape}')
print('\nSample column names:', cases_raw.columns[:5].tolist())
cases_raw.iloc[:3, :6]

## 3. Data Preprocessing

In [ ]:
def preprocess(df):
    """
    The CONVENIENT format:
      - Row 0 : Province/State sub-region labels (metadata, skip)
      - Rows 1+ : date string in col-0, daily new counts in remaining cols
    Steps:
      1. Drop metadata row; rename first column to 'Date'; parse as datetime.
      2. Convert all count columns to numeric.
      3. Strip .N suffixes (e.g. 'Australia.1' -> 'Australia') and sum
         sub-national columns back into their parent country.
      4. Sort chronologically, forward-fill sparse gaps.
    """
    dates_col  = df.iloc[1:, 0].copy()
    data_part  = df.iloc[1:, 1:].copy()
    countries  = [c.split('.')[0].strip() for c in df.columns[1:]]
    data_part.columns = countries
    data_part  = data_part.apply(pd.to_numeric, errors='coerce')
    # Sum sub-national duplicate columns
    data_part  = data_part.T.groupby(level=0).sum().T
    data_part['Date'] = pd.to_datetime(dates_col.values, format='%m/%d/%y')
    data_part  = data_part.sort_values('Date').reset_index(drop=True)
    date_saved = data_part['Date']
    data_part  = data_part.drop(columns='Date').apply(pd.to_numeric, errors='coerce')
    data_part  = data_part.ffill().dropna(axis=1, how='all')
    data_part['Date'] = date_saved
    return data_part

daily_cases_df  = preprocess(cases_raw)
daily_deaths_df = preprocess(deaths_raw)

print(f'Daily cases  shape : {daily_cases_df.shape}')
print(f'Daily deaths shape : {daily_deaths_df.shape}')
print(f'Date range : {daily_cases_df["Date"].min().date()} -> {daily_cases_df["Date"].max().date()}')

In [ ]:
cc = [c for c in daily_cases_df.columns  if c != 'Date']
dc = [c for c in daily_deaths_df.columns if c != 'Date']
dates = daily_cases_df['Date']

# Clip negatives (reporting corrections), then cumsum for cumulative totals
daily_cases_global  = daily_cases_df[cc].sum(axis=1).clip(lower=0).astype(float)
daily_deaths_global = daily_deaths_df[dc].sum(axis=1).clip(lower=0).astype(float)
global_cases  = daily_cases_global.cumsum()
global_deaths = daily_deaths_global.cumsum()
roll_cases    = daily_cases_global.rolling(7, min_periods=1).mean()
roll_deaths   = daily_deaths_global.rolling(7, min_periods=1).mean()

cfr = global_deaths.iloc[-1] / global_cases.iloc[-1] * 100
print(f'Total cumulative cases  : {global_cases.iloc[-1]/1e6:.1f}M')
print(f'Total cumulative deaths : {global_deaths.iloc[-1]/1e6:.2f}M')
print(f'Case Fatality Rate      : {cfr:.2f}%')
print(f'Peak single-day cases   : {daily_cases_global.max()/1e6:.2f}M')
print(f'Peak single-day deaths  : {daily_deaths_global.max():,.0f}')

In [ ]:
# ── Build global totals (sum across all countries) ────────────────────────────
count_cols_c = [c for c in cases.columns  if c != 'Date']
count_cols_d = [c for c in deaths.columns if c != 'Date']

global_cases  = cases[count_cols_c].sum(axis=1)
global_deaths = deaths[count_cols_d].sum(axis=1)
dates         = cases['Date']

# Daily new cases / deaths (first-difference of cumulative)
daily_cases  = global_cases.diff().clip(lower=0).fillna(0)
daily_deaths = global_deaths.diff().clip(lower=0).fillna(0)

# 7-day rolling averages
roll_cases  = daily_cases.rolling(7, min_periods=1).mean()
roll_deaths = daily_deaths.rolling(7, min_periods=1).mean()

print('Global totals built.')
print(f'Peak single-day cases : {daily_cases.max():,.0f}')
print(f'Total confirmed cases : {global_cases.iloc[-1]:,.0f}')
print(f'Total deaths          : {global_deaths.iloc[-1]:,.0f}')

## 4. Exploratory Data Analysis

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Metric'              : ['Total Confirmed Cases', 'Total Deaths',
                              'Case Fatality Rate (%)', 'Peak Daily Cases',
                              'Peak Daily Deaths'],
    'Value'               : [
        f"{global_cases.iloc[-1]:,.0f}",
        f"{global_deaths.iloc[-1]:,.0f}",
        f"{global_deaths.iloc[-1]/global_cases.iloc[-1]*100:.2f}",
        f"{daily_cases.max():,.0f}",
        f"{daily_deaths.max():,.0f}"
    ]
})
print(summary.to_string(index=False))

In [ ]:
# ── Descriptive stats for major countries ─────────────────────────────────────
major = ['US', 'India', 'Brazil', 'France', 'Germany', 'United Kingdom']
major = [m for m in major if m in cases.columns]   # keep only those present

desc = cases[major].describe().applymap(lambda x: f'{x:,.0f}')
print('Cumulative confirmed cases – descriptive statistics:')
desc

In [ ]:
# ── Plot 1 : Global cumulative cases & deaths ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].fill_between(dates, global_cases / 1e6, alpha=0.3, color='steelblue')
axes[0].plot(dates, global_cases / 1e6, color='steelblue', lw=1.5)
axes[0].set_title('Global Cumulative Confirmed Cases', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cases (millions)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

axes[1].fill_between(dates, global_deaths / 1e6, alpha=0.3, color='crimson')
axes[1].plot(dates, global_deaths / 1e6, color='crimson', lw=1.5)
axes[1].set_title('Global Cumulative Deaths', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Deaths (millions)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig1_cumulative.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2 : Daily new cases & 7-day rolling average ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(dates, daily_cases / 1e6, color='steelblue', alpha=0.4, width=1, label='Daily')
axes[0].plot(dates, roll_cases / 1e6, color='navy', lw=2, label='7-day avg')
axes[0].set_title('Daily New Cases (Global)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cases (millions)')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

axes[1].bar(dates, daily_deaths / 1e3, color='crimson', alpha=0.4, width=1, label='Daily')
axes[1].plot(dates, roll_deaths / 1e3, color='darkred', lw=2, label='7-day avg')
axes[1].set_title('Daily New Deaths (Global)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Deaths (thousands)')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig2_daily.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3 : Top-10 countries by total cases (bar chart) ─────────────────────
top10 = cases[count_cols_c].iloc[-1].sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top10.index[::-1], top10.values[::-1] / 1e6,
               color=sns.color_palette('tab10', 10))
ax.set_xlabel('Total Confirmed Cases (millions)')
ax.set_title('Top 10 Countries by Total Confirmed Cases', fontsize=13, fontweight='bold')
for bar, val in zip(bars, top10.values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.1f}M', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig3_top10.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 4 : Correlation matrix (major countries, daily new cases) ────────────
corr_df = cases[major].diff().clip(lower=0).fillna(0).corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=ax, mask=mask,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation of Daily New Cases (Major Countries)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig4_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Development – SIR Compartmental Model

The **SIR model** partitions the population into:
- **S** – Susceptible
- **I** – Infectious
- **R** – Removed (recovered + deceased)

$$\frac{dS}{dt} = -\beta \frac{SI}{N}, \quad \frac{dI}{dt} = \beta \frac{SI}{N} - \gamma I, \quad \frac{dR}{dt} = \gamma I$$

Parameters β (transmission rate) and γ (recovery rate) are estimated by minimising RMSE on the first 200 days of global data.

In [ ]:
# ── SIR model definition ──────────────────────────────────────────────────────
def sir_model(y, t, N, beta, gamma):
    S, I, R = y
    dS = -beta * S * I / N
    dI =  beta * S * I / N - gamma * I
    dR =  gamma * I
    return dS, dI, dR


def fit_sir(cumulative_infected, N, n_days=200):
    """Fit β and γ to the first `n_days` of the cumulative infected series."""
    obs = cumulative_infected[:n_days].values
    t   = np.arange(n_days)

    # I(0) = first observed infected, R(0) = 0
    I0 = obs[0] if obs[0] > 0 else 1
    R0_init = 0
    S0 = N - I0 - R0_init
    y0 = S0, I0, R0_init

    def loss(params):
        beta, gamma = params
        if beta <= 0 or gamma <= 0:
            return 1e12
        sol = odeint(sir_model, y0, t, args=(N, beta, gamma))
        I_pred = sol[:, 1] + sol[:, 2]   # cumulative = I + R
        return np.sqrt(np.mean((I_pred - obs) ** 2))

    result = minimize(loss, x0=[0.3, 0.1],
                      bounds=[(1e-4, 10), (1e-4, 10)],
                      method='L-BFGS-B')
    return result.x   # beta, gamma


# ── Fit on global data ────────────────────────────────────────────────────────
N_global = 7_900_000_000   # approximate world population
N_DAYS   = 200

print(f'Fitting SIR on first {N_DAYS} days of global data …')
beta_fit, gamma_fit = fit_sir(global_cases, N_global, N_DAYS)

R0_basic = beta_fit / gamma_fit
print(f'\nEstimated β (transmission rate) : {beta_fit:.4f}')
print(f'Estimated γ (recovery rate)     : {gamma_fit:.4f}')
print(f'Basic reproduction number R₀    : {R0_basic:.2f}')

In [ ]:
# ── Generate SIR trajectory for N_DAYS ───────────────────────────────────────
I0  = global_cases.iloc[0] if global_cases.iloc[0] > 0 else 1
R0v = 0
S0  = N_global - I0 - R0v
y0  = S0, I0, R0v
t   = np.arange(N_DAYS)

sol     = odeint(sir_model, y0, t, args=(N_global, beta_fit, gamma_fit))
S_pred  = sol[:, 0]
I_pred  = sol[:, 1]
R_pred  = sol[:, 2]
cum_pred = I_pred + R_pred   # total ever-infected (≈ cumulative cases)

obs = global_cases.iloc[:N_DAYS].values
print('SIR trajectory generated.')

## 6. Model Evaluation

In [ ]:
mae  = mean_absolute_error(obs, cum_pred)
mse  = mean_squared_error(obs, cum_pred)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((obs - cum_pred) / (obs + 1))) * 100

print('═' * 40)
print('   SIR Model Evaluation Metrics')
print('═' * 40)
print(f'  MAE  : {mae:>18,.0f}')
print(f'  MSE  : {mse:>18,.0f}')
print(f'  RMSE : {rmse:>18,.0f}')
print(f'  MAPE : {mape:>17.2f} %')
print('═' * 40)

## 7. Visualization – Model vs. Observed

In [ ]:
# ── Plot 5 : SIR model vs observed cumulative cases ───────────────────────────
fit_dates = dates.iloc[:N_DAYS]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(fit_dates, obs / 1e6, 'o', ms=2, color='steelblue', label='Observed (cumulative)', alpha=0.7)
ax.plot(fit_dates, cum_pred / 1e6, '-', lw=2.5, color='darkorange', label='SIR model fit')
ax.set_title(f'SIR Model vs. Observed Global Cases  (first {N_DAYS} days)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Cases (millions)')
ax.set_xlabel('Date')
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

# Annotate R₀
ax.text(0.02, 0.92, f'R₀ = {R0_basic:.2f}  |  β = {beta_fit:.4f}  |  γ = {gamma_fit:.4f}',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig5_sir_fit.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 6 : SIR compartments over time ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(fit_dates, S_pred / 1e9, label='Susceptible (S)', color='steelblue', lw=2)
ax.plot(fit_dates, I_pred / 1e6, label='Infectious (I)  ×1e6', color='darkorange', lw=2)
ax.plot(fit_dates, R_pred / 1e6, label='Removed (R)  ×1e6',   color='seagreen',   lw=2)
ax.set_title('SIR Compartment Trajectories', fontsize=13, fontweight='bold')
ax.set_ylabel('Population (billions for S, millions for I & R)')
ax.set_xlabel('Date')
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig6_sir_compartments.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 7 : Residuals (observed – predicted) ─────────────────────────────────
residuals = obs - cum_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fit_dates, residuals / 1e6, color='purple', lw=1.2)
axes[0].axhline(0, ls='--', color='black', lw=1)
axes[0].set_title('Residuals Over Time', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Residual (millions)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

axes[1].hist(residuals / 1e6, bins=30, color='purple', alpha=0.7, edgecolor='white')
axes[1].set_title('Residual Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Residual (millions)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig7_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n── All analysis complete. Figures saved to /mnt/user-data/outputs/ ──')